## Importing Packages

python scripts/reformat_dasa.py --datadir data --rename data/rename_columns.xlsx --correction data/fix_values.xlsx --output combined_dasa.tsv

In [2]:
import pandas as pd
import os
import re

## Loading Data

In [3]:
def load_table(file, separator=None):
    df = ''
    if str(file).split('.')[-1] == 'tsv':
        separator = '\t' if separator is None else separator
        df = pd.read_csv(file, encoding='latin-1', sep=separator, dtype='str')
    elif str(file).split('.')[-1] == 'csv':
        separator = ',' if separator is None else separator
        df = pd.read_csv(file, encoding='latin-1', sep=separator, dtype='str')
    elif str(file).split('.')[-1] in ['xls', 'xlsx']:
        df = pd.read_excel(file, index_col=None, header=0, sheet_name=0, dtype='str')
        df.fillna('', inplace=True)
    else:
        print('Wrong file format. Compatible file formats: TSV, CSV, XLS, XLSX')
        exit()
    return df

In [4]:
BASE_PATH = '../data/DASA/'
dfs = []

for file in os.listdir(BASE_PATH):
    if file.endswith(('.xlsx', '.tsv')):
        print(file)
        df = load_table(BASE_PATH + file, separator='\t')
        dfs.append(df)

20220712_dados Dasa suspect_omicron_2022-07-03_2022-07-09-thermo para ITpS.xlsx
20230419_Dados Dasa patogenos_respiratorios_2023-04-09_2023-04-15 para ITpS_SE15.xlsx
20230713_Dados Dasa patogenos_respiratorios_2023-07-02_2023-07-08 para ITpS_SE27.xlsx
20220614_Dados Dasa patogenos_respiratorios_2022_06_05-2022_06_11 para ITpS.xlsx
20220621_Dados Dasa patogenos_respiratorios_2022-06-12_2022-06-18.xlsx
20221101_dados Dasa suspect_omicron_2022-10-09_2022-10-15-thermo para ITpS_SE41.xlsx
20220517_Dados Dasa patogenos_respiratorios_2022_05_08-2022_05_14 para ITpS.xlsx
20220722_Dados Dasa patogenos_respiratorios_2022-07-10_2022-07-16 para ITpS.xlsx
20220722_dados Dasa suspect_omicron_2022-07-10_2022-07-16-thermo para ITpS.xlsx
20221101_Dados Dasa suspect_omicron_2022-10-23_2022-10-29-thermo ITpS_SE43.xlsx
20220928_Dados Dasa patogenos_respiratorios_2022-09-18_2022-09-24 para ITpS.xlsx
20230529_Dados Dasa suspect_omicron_2023-05-14_2023-05-20-thermo para ITpS_SE20.xlsx
20221005_dados Dasa sus

## Codigo

In [5]:
codigo_dfs = [ df for df in dfs if 'codigo' in df.columns.tolist() ]

In [6]:
TEST_NAME = "test_4"

In [7]:
codigo_df = pd.concat(codigo_dfs)
codigo_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 131680 entries, 0 to 2391
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   ano_mes           131680 non-null  object
 1   data_exame        131680 non-null  object
 2   nome_pac          3840 non-null    object
 3   sexo              131680 non-null  object
 4   idade             131680 non-null  object
 5   cidade            128648 non-null  object
 6   uf                131680 non-null  object
 7   telefone          3840 non-null    object
 8   email             3840 non-null    object
 9   codigorequisicao  131680 non-null  object
 10  codigo            131680 non-null  object
 11  positivo          131680 non-null  object
dtypes: object(12)
memory usage: 13.1+ MB


In [8]:
DROP_COLUMNS = ['email', 'telefone', 'nome_pac', 'ano_mes']
codigo_df = codigo_df.drop(columns=DROP_COLUMNS)

### Valores únicos

In [9]:
codigo_df['codigo'].unique()

array(['FLUA', 'FLUB', 'VSR', 'COVID'], dtype=object)

In [10]:
codigo_df['positivo'].unique()

array(['1', '0'], dtype=object)

In [11]:
codigo_df['codigorequisicao'].replace(re.compile('[0-9]'), 'x').unique()

array(['xxxxxxxxxxxx', ''], dtype=object)

In [12]:
(
    codigo_df
    .query("codigorequisicao != ''")
    
    # query where 'cidade' is not NaN
    .query("cidade == cidade")
    .assign(duplicated=lambda df: df.duplicated(subset=['codigorequisicao']))
    .groupby('codigorequisicao')
    .agg({'duplicated': 'sum'})
    .assign(duplicated=lambda df: df['duplicated'] + 1)
    .sort_values('duplicated', ascending=False)
    ['duplicated']
    .unique()
)

array([4])

In [34]:
codigo_df.query("codigorequisicao == '934401167060'")

,date_testing,sex,age,location,state,codigorequisicao,codigo,result
736,2022-03-08,F,64,NaN,SP,934401167060,FLUA,0
737,2022-03-08,F,64,NaN,SP,934401167060,FLUB,0
738,2022-03-08,F,64,NaN,SP,934401167060,VSR,0
739,2022-03-08,F,64,NaN,SP,934401167060,COVID,0
151,2022-03-08,F,64,SAO PAULO,SP,934401167060,FLUA,0
833,2022-03-08,F,64,SAO PAULO,SP,934401167060,FLUB,0
1515,2022-03-08,F,64,SAO PAULO,SP,934401167060,VSR,0
2197,2022-03-08,F,64,SAO PAULO,SP,934401167060,COVID,0


Problema:
O campo codigorequisicao tem valores faltantes = ''

Solução:
Drop de linhas com valores faltantes

Problema:

O campo codigorequisicao se repete 4 ou 8 vezes nos arquivos.
A repetição de 4x é devido aos 4 tipos de patógenos capturados, enquanto a de 8x parece ser um erro de duplicação.

O erro de duplicação tem uma característica marcante: aparentemente, toda entrada possui o campo 'cidade' como NaN. Ou seja, uma possivel solução para remover as duplicatas é remover linhas que possuem NaN em 'cidade'.


Solução: 

drop_duplicates( subset = ['codigorequisicao', 'codigo'] )

.query("cidade == cidade")  # Um NaN não é igual a ele mesmo, então essa query remove NaNs

In [14]:
codigo_df['positivo'] = codigo_df['positivo'].astype(int)

codigo_df = (
    codigo_df
    .rename(
        columns={
            'data_exame': 'date_testing',
            'positivo': 'result',
            'sexo': 'sex',
            'idade': 'age',
            'cidade': 'location',
            'uf': 'state',
        }
    )
)

In [15]:
codigo_df

,date_testing,sex,age,location,state,codigorequisicao,codigo,result
0,2023-04-11 00:00:00,M,31,BRASILIA,DF,804301974351,FLUA,1
1,2023-04-13 00:00:00,M,53,CASCAVEL,PR,541619041404,FLUA,1
2,2023-04-12 00:00:00,M,3,CASCAVEL,PR,926027817672,FLUA,1
3,2023-04-12 00:00:00,F,52,SAO PAULO,SP,840501148123,FLUA,1
4,2023-04-12 00:00:00,F,77,RIO DE JANEIRO,RJ,952900082028,FLUA,1
...,...,...,...,...,...,...,...,...
2387,2022-03-25,F,58,BRASILIA,DF,802200132711,COVID,0
2388,2022-03-27,F,57,TAGUATINGA,DF,804400986712,COVID,0
2389,2022-03-23,F,2,BRASILIA,DF,882901146941,COVID,0
2390,2022-03-25,F,0,SAO PAULO,SP,899701314661,COVID,0


## Gene S

In [16]:
gene_s_dfs = []
for df in dfs:

    if 'Gene S' not in df.columns.tolist():
        continue

    rename_dict = {}
    df_columns = df.columns.tolist() 
    if 'resultado' not in df_columns:
        
        if 'resultado_norm' in df_columns:
            rename_dict['resultado_norm'] = 'resultado'
        elif 'resultado_original' in df_columns:
            # Nenhum caso encontrado
            rename_dict['resultado_original'] = 'resultado'

    if 'requisicao' not in df_columns:
        if 'codigo_externo_do_paciente' in df_columns:
            rename_dict['codigo_externo_do_paciente'] = 'requisicao'

    gene_s_dfs.append(df.rename(columns=rename_dict, errors='ignore'))

### Valores únicos

- Regra: Atribuir 'thermo' a test_kit
- Regra: Atribuir '' a cidade_norm, uf_norm caso o valor seja 'SEM CIDADE', 'MUDOU', 'NAO_INFORMADO' ou 'NAOINFORMADO'
- Regra: Atribuir NT a todos os patógenos que não sejam SC2
- Regra: Atribuir '' a Gene N, Gene ORF e GENE S caso o resultado seja Neg
- Regra: Transformar o valor de Gene N, Gene ORF e GENE S em string com 1 casa decimal


In [17]:
# DÚVIDA
# for idx, row in dfL.iterrows():
#     result = dfL.loc[idx, 'resultado']
#     if result == 'NAO DETECTADO':
#         dfL.loc[idx, 'Gene N'] = ''
#         dfL.loc[idx, 'Gene ORF'] = ''
#         dfL.loc[idx, 'Gene S'] = ''
#     else: # if not reported
#         result = 'NA'
#         dfL.loc[idx, 'Gene N'] = str(round(float(dfL.loc[idx, 'Gene N']), 1))
#         dfL.loc[idx, 'Gene ORF'] = str(round(float(dfL.loc[idx, 'Gene ORF']), 1))
#         dfL.loc[idx, 'Gene S'] = str(round(float(dfL.loc[idx, 'Gene S']), 1))


In [18]:
gene_df = pd.concat(gene_s_dfs)
gene_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 407665 entries, 0 to 1816
Data columns (total 62 columns):
 #   Column                           Non-Null Count   Dtype 
---  ------                           --------------   ----- 
 0   requisicao                       310536 non-null  object
 1   data                             407665 non-null  object
 2   data_de_nascimento               364174 non-null  object
 3   idade                            404276 non-null  object
 4   sexo                             380901 non-null  object
 5   teste                            407665 non-null  object
 6   data_do_resultado                165884 non-null  object
 7   data_de_liberacao                204885 non-null  object
 8   resultado_original               118319 non-null  object
 9   cidade_norm                      407665 non-null  object
 10  uf_norm                          407665 non-null  object
 11  regiao_norm                      407665 non-null  object
 12  resultado         

In [19]:
gene_df.columns

Index(['requisicao', 'data', 'data_de_nascimento', 'idade', 'sexo', 'teste',
       'data_do_resultado', 'data_de_liberacao', 'resultado_original',
       'cidade_norm', 'uf_norm', 'regiao_norm', 'resultado', 'regiao_norm2',
       'positivo', 'Gene N', 'Gene ORF', 'Gene S', 's_dropout', 'material',
       'resultado_norm', 'plataforma', 'executor', 'data_prometida',
       'ponto_de_atendimento', 'bairro', 'cep', 'codigo_externo_do_paciente',
       'codigo_externo_da_visita', 'divisao', 'cod_hospitalar', 'tipo_de_area',
       'localizacao', 'paciente', 'telefone', 'cod_amostra',
       'posicao_no_container', 'nome_do_container', 'numero_do_container',
       'status', 'nivel_de_liberacao', 'responsavel_pela_liberacao',
       'convenio', 'numero_do_convenio', 'amostras_primarias',
       'cliente_de_laboratorio_apoiador', 'liberado_em_atraso', 'chave',
       'unidade_medida', 'cep_dasa', 'gliese_cidade', 'dasa_cidade',
       'apoio_cidade', 'gliese_uf', 'dasa_uf', 'apoio_uf', 'ce

In [20]:
[
    "requisicao",
    "data",
    "data_de_nascimento",
    "idade",
    "sexo",
    "resultado",
    "resultado_norm",
    "resultado_original",
    "cidade_norm",
    "uf_norm",
    "positivo",
    "Gene N",
    "Gene ORF",
    "Gene S",
]

['requisicao',
 'data',
 'data_de_nascimento',
 'idade',
 'sexo',
 'resultado',
 'resultado_norm',
 'resultado_original',
 'cidade_norm',
 'uf_norm',
 'positivo',
 'Gene N',
 'Gene ORF',
 'Gene S']

In [21]:
gene_df['positivo']

0        True
1       False
2        True
3       False
4       False
        ...  
1812     True
1813     True
1814    False
1815    False
1816    False
Name: positivo, Length: 407665, dtype: object

In [22]:
{
    'POSITIVO (SEM CT)': 'Pos',
    'NEGATIVO': 'Neg',
    'DETECTADO': 'Pos',
    'NÃO DETECTADO': 'Neg',
}

gene_df['resultado'].unique()

array(['POSITIVO (SEM CT)', 'NEGATIVO', 'DETECTADO', 'NAO DETECTADO'],
      dtype=object)

In [23]:
gene_df['sexo'].unique()

array(['MASCULINO', 'FEMININO', 'INDETERMINADO', 'None', nan, ''],
      dtype=object)

In [24]:
gene_df['teste'].unique()

array(['DETECCAO QUALITATIVA DE CORONAVIRUS (2019-NCOV)'], dtype=object)

In [25]:
gene_df['requisicao'].replace(re.compile('[0-9]'), 'x').unique()

array(['xxxxxxxxxx', 'xxxxxxxxx', 'xxxxxxxx', 'xxxxx', 'xxxxxxx',
       'xxxxxx', 'xxxx.xxxx.xxxx', nan, 'xxxx', 'xxx'], dtype=object)

In [31]:
gene_df['idade'].fillna('-1').unique()

array(['32', '41', '25', '22', '45', '34', '30', '54', '88', '28', '33',
       '7', '1', '40', '46', '26', '59', '65', '73', '11', '27', '35',
       '42', '10', '38', '37', '58', '36', '21', '19', '29', '50', '83',
       '24', '39', '4', '8', '44', '79', '53', '20', '2', '66', '55',
       '16', '31', '62', '47', '48', '56', '64', '43', '52', '3', '0',
       '14', '23', '51', '49', '67', '9', '70', '6', '63', '86', '72',
       '12', '18', '60', '13', '17', '74', '5', '68', '75', '61', '57',
       '69', '71', '94', '96', '82', '81', '15', '91', '84', '85', '77',
       '92', '80', '76', '122', '89', '93', '78', '90', '87', '95', '98',
       '97', '102', '99', '103', '101', '1993', '100', '104', '120', '',
       '807', '2012', '107', '822', '1713', '123', '106', '105', '114',
       '945', '544', '170'], dtype=object)

In [26]:
(
    gene_df
    
    .assign(duplicated=lambda df: df.duplicated(subset=['requisicao']))
    .groupby('requisicao')
    .agg({'duplicated': 'sum'})
    .assign(duplicated=lambda df: df['duplicated'] + 1)
    .sort_values('duplicated', ascending=False)
    .query('duplicated > 1')
    # ['duplicated']
    # .unique()
)

,duplicated
requisicao,
7662439349,24
7660500854,21
1223194821,15
6163693888,15
6180840616,12
...,...
6173397367,2
7661248824,2
59010902,2


In [ ]:
12222

In [27]:
gene_df['idade']

0        32
1        41
2        25
3        22
4        45
       ... 
1812    NaN
1813    NaN
1814    NaN
1815    NaN
1816    NaN
Name: idade, Length: 407665, dtype: object

In [28]:
gene_df.query("requisicao == '7677105586'")

,requisicao,data,data_de_nascimento,idade,sexo,teste,data_do_resultado,data_de_liberacao,resultado_original,cidade_norm,...,apoio_cidade,gliese_uf,dasa_uf,apoio_uf,cep_municipio,cep_uf,cep_municipio_dasa,cep_uf_dasa,cidade_tratado,uf_tratado
1030,7677105586,2022-10-10 09:46:50+00:00,1974-10-14,47,MASCULINO,DETECCAO QUALITATIVA DE CORONAVIRUS (2019-NCOV),2022-10-11 19:55:37+00:00,NaN,NaN,SAOPAULO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2612,7677105586,2022-08-21 08:07:38+00:00,1974-10-14,47,MASCULINO,DETECCAO QUALITATIVA DE CORONAVIRUS (2019-NCOV),2022-08-22 19:02:51+00:00,NaN,NaN,SAOPAULO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Problema: Duplicatas de id requisicao

Solução: Remover duplicatas com drop duplicates